# Ad-moment scorer: training notebook

**Preprocess first** (CPU, needs Bronze logs): [`build_policy_dataset.ipynb`](build_policy_dataset.ipynb).
This notebook only trains. It loads `outputs/silver/turns.csv` and `outputs/anchor/human_anchor.csv`.

**What it trains.** A serving-policy component \(\pi\): given the conversation up to the current user utterance, output **one scalar** \(m\in[0,1]\), how good this moment is for *some* advertisement. Presentation \(\lambda\) (implicit / explicit) is neither an input nor an output. No person covariates, no EEG, no task ids at inference.

**Inputs \(X\).** For user turn \(k\):
\[
x_k = \text{serialise}\big([\texttt{TURN}\ \min(k,8)],\ u_{k-3}, a_{k-3}, \ldots, u_{k}\big) \in \mathcal V^{\le 512},
\qquad
z_k = \big(\tfrac{\min(k,8)}{8},\ \text{fit}_k \in[0,1]\ \text{or missing}\big)
\]
The encoder sees only the text \(x_k\); \(z_k\) enters the calibration layer.

**Targets \(Y\).** Two layers, never collapsed, plus one optional diagnostic:

* Dense, weak, turn grain (judge): \(\mathbf r_k = (r^{\text{need}}, r^{\text{ready}}, r^{\text{safe}}, r^{\text{nodeg}}) \in [0,1]^4\), \(\bar r_k = \tfrac14\sum_j r^{(j)}_k\).
  The encoder is a **4-output regression** \(\hat{\mathbf r}_k = \sigma(W h_\theta(x_k) + b)\) trained with soft-target BCE.
* Sparse, human, ad grain (216 rows, frozen anchor): \(U_i = \tfrac13(\Delta_{\text{cred}} + \Delta_{\text{trust}} - \Delta_{\text{manip}})\) person-centred against the participant's own no-ad conversation and residualised on \(\lambda\) and task; \(y_i = \mathbb 1[U_i^{\text{resid}} \ge 0]\).
  The calibrator is a logistic \(m_k = \sigma(\alpha^\top[\hat{\bar r}_k, z_k] + \beta)\) fitted leave-one-participant-out.
* Optional diagnostic (not the gate): frozen encoder \(\to\) 6 UX deltas on those 216 rows.

**Final output.** `moment_score` \(= m_k\), plus the four rubric heads, plus a rule veto (governance \(\Gamma\), not learned), plus optional `ux_hat`.

**Environments.** Set `AD_MOMENT_ENV` or let cell 1 detect: `atlas` (2×A100), `laptop16` (16 GB), `colab` (T4), `kaggle` (P100/T4), `smoke` / `cpu`. On Colab/Kaggle: zip `outputs/{silver,anchor}` from the preprocess notebook (not `notebook_runs`).


In [ ]:
# ---- 0. Configuration ------------------------------------------------------
import os, sys, json, math, random, re, time, zipfile, hashlib
from pathlib import Path

SEED = 13
# smoke | full  — full is the default; env profiles then shrink it to fit the GPU
RUN_MODE = os.environ.get("AD_MOMENT_RUN_MODE", "full")

SMOKE = dict(
    judge_model="Qwen/Qwen3-0.6B", n_external=60, external_source="oasst1",
    encoder="Thrad/thrad-bert-conversation-classifier", baselines=[],
    epochs=1, batch_size=16, lr=3e-5, max_len=256, cv_folds=2, max_train_steps=40,
    ablation_windows=[4], ablation_external=[True], judge_audit_n=20, judge_batch=8,
    train_ux_head=False,
)
FULL = dict(
    judge_model="Qwen/Qwen3-4B-Instruct-2507", n_external=8000, external_source="wildchat",
    encoder="Thrad/thrad-bert-conversation-classifier", baselines=["BAAI/bge-small-en-v1.5"],
    epochs=3, batch_size=32, lr=3e-5, max_len=512, cv_folds=5, max_train_steps=None,
    ablation_windows=[1, 2, 4], ablation_external=[True, False], judge_audit_n=200, judge_batch=24,
    train_ux_head=True,
)

# Env profiles overlay FULL (or SMOKE). Detected in the next cell unless AD_MOMENT_ENV is set.
ENV_OVERLAY = {
    "atlas":    dict(),  # 40 GB A100: FULL as written
    "laptop16": dict(n_external=2000, batch_size=16, max_len=384, judge_batch=4, judge_audit_n=80,
                     ablation_windows=[4], ablation_external=[True]),
    "colab":    dict(n_external=4000, batch_size=16, judge_batch=8, judge_audit_n=120,
                     ablation_windows=[2, 4], ablation_external=[True]),
    "kaggle":   dict(n_external=4000, batch_size=16, judge_batch=8, judge_audit_n=120,
                     external_source="oasst1",  # open; WildChat needs a token before Kaggle goes offline
                     ablation_windows=[2, 4], ablation_external=[True]),
    "cpu":      dict(**SMOKE),
    "smoke":    dict(**SMOKE),
}

# Governance vetoes (Gamma). Rule, not learned. ThradBERT runtime labels.
VETO_GENRES = {"other_obscene_or_illegal", "relationships_and_personal_reflection"}
MAX_PREFIX_TURNS = 4
ROLE_TAGS = {"user": "[USER]", "assistant": "[ASSISTANT]"}
UX_COLS = ["delta_credibility", "delta_trust", "delta_manipulation",
           "delta_helpfulness", "notice_sponsored", "recall_memory"]

UX_REPORT = {"trained": False}  # overwritten in 8b if the diagnostic head runs

random.seed(SEED)
print("RUN_MODE before env detect:", RUN_MODE)


In [ ]:
# ---- 1. Environment --------------------------------------------------------
def _pip(*pkgs):
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

try:
    import torch, transformers, datasets, accelerate, sklearn, pandas, scipy
except ImportError:
    _pip("torch", "transformers>=4.45", "datasets", "accelerate", "scikit-learn", "pandas", "scipy")
    import torch, transformers, datasets, accelerate, sklearn, pandas, scipy

import numpy as np, pandas as pd
import torch, torch.nn as nn
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP_DTYPE = torch.bfloat16 if (DEVICE == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16
torch.manual_seed(SEED); np.random.seed(SEED)
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

def detect_env() -> str:
    forced = os.environ.get("AD_MOMENT_ENV")
    if forced:
        return forced
    if RUN_MODE == "smoke":
        return "smoke"
    if IN_COLAB:
        return "colab"
    if IN_KAGGLE:
        return "kaggle"
    if DEVICE == "cpu":
        return "cpu"
    gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gb >= 30:
        return "atlas"
    if gb >= 12:
        return "laptop16"
    return "cpu"

ENV = detect_env()
if ENV in ("smoke", "cpu") and os.environ.get("AD_MOMENT_RUN_MODE") is None:
    RUN_MODE = "smoke"
CFG = dict(SMOKE if RUN_MODE == "smoke" or ENV in ("smoke", "cpu") else FULL)
CFG.update(ENV_OVERLAY.get(ENV, {}))
if os.environ.get("AD_MOMENT_JUDGE"):
    CFG["judge_model"] = os.environ["AD_MOMENT_JUDGE"]
if os.environ.get("AD_MOMENT_N_EXTERNAL"):
    CFG["n_external"] = int(os.environ["AD_MOMENT_N_EXTERNAL"])

if DEVICE == "cuda":
    pdev = torch.cuda.get_device_properties(0)
    print(f"GPU: {pdev.name}, {pdev.total_memory/1e9:.1f} GB, amp dtype {AMP_DTYPE}")
else:
    print("CPU only — smoke profile. A full judge on CPU is hours.")
print("ENV:", ENV, "RUN_MODE:", RUN_MODE)
print(json.dumps(CFG, indent=2))
print("""
VRAM cheat-sheet (sequential: delete the judge before the encoder):
  67M DistilBERT fine-tune   2–3 GB
  Qwen3-0.6B judge (smoke)   ~2 GB
  Qwen3-4B judge fp16        8–10 GB   (atlas / colab T4 / 16 GB laptop)
  Do not load 4B and the encoder at the same time.
""")


In [ ]:
# ---- 2. Locate the Silver + anchor tables ----------------------------------
CANDIDATES = [Path("outputs"), Path("analysis/policy/outputs"), Path("../analysis/policy/outputs"),
              Path("/content/outputs"), Path("/content/drive/MyDrive/ad_moment/outputs")]
if "__file__" not in globals():
    CANDIDATES.insert(0, Path.cwd() / "outputs")

def locate_data() -> Path:
    for c in CANDIDATES:
        if (c / "silver" / "turns.csv").exists() and (c / "anchor" / "human_anchor.csv").exists():
            return c.resolve()
    if IN_COLAB:
        from google.colab import files
        print("Upload a zip of analysis/policy/outputs (must contain silver/turns.csv and anchor/human_anchor.csv)")
        up = files.upload()
        for name in up:
            with zipfile.ZipFile(name) as z:
                z.extractall("/content")
        for c in [Path("/content/outputs"), Path("/content/analysis/policy/outputs")]:
            if (c / "silver" / "turns.csv").exists():
                return c
    raise FileNotFoundError(
        "Silver/anchor missing. On a machine with the repo logs run "
        "analysis/policy/build_policy_dataset.ipynb (or the two .py scripts), "
        "then zip outputs/{silver,anchor} and upload that zip here."
    )

DATA = locate_data()
WORK = DATA / "notebook_runs" / RUN_MODE
WORK.mkdir(parents=True, exist_ok=True)
turns = pd.read_csv(DATA / "silver" / "turns.csv")
conversations = pd.read_csv(DATA / "silver" / "conversations.csv")
anchor = pd.read_csv(DATA / "anchor" / "human_anchor.csv")
anchor_hash = hashlib.sha256((DATA / "anchor" / "human_anchor.csv").read_bytes()).hexdigest()[:16]
print(DATA); print("turns", turns.shape, turns.source_set.value_counts().to_dict())
print("anchor", anchor.shape, "sha256_16", anchor_hash, "good_moment_rate", round(anchor.good_moment_human.mean(), 3))


In [ ]:
# ---- 3. Serialisation contract (identical to build_policy_silver.py and serving) ---
def serialise_prefix(messages, turn_index, max_turns=MAX_PREFIX_TURNS):
    user_positions = [i for i, m in enumerate(messages) if m["role"] == "user"]
    if len(user_positions) > max_turns:
        messages = messages[user_positions[-max_turns]:]
    parts = [f"[TURN {min(turn_index, 8)}]"]
    for m in messages:
        parts.append(f"{ROLE_TAGS[m['role']]} {m['content'].strip()}")
    return "\n".join(parts)

def reserialise(prefix_text, max_turns):
    # rebuild a shorter window from the stored 4-turn serialisation
    head, *body = prefix_text.split("\n")
    turn_index = int(re.match(r"\[TURN (\d+)\]", head).group(1))
    msgs, cur = [], None
    for line in body:
        if line.startswith("[USER] "):
            cur = {"role": "user", "content": line[len("[USER] "):]}; msgs.append(cur)
        elif line.startswith("[ASSISTANT] "):
            cur = {"role": "assistant", "content": line[len("[ASSISTANT] "):]}; msgs.append(cur)
        elif cur is not None:
            cur["content"] += "\n" + line
    return serialise_prefix(msgs, turn_index, max_turns)

assert reserialise(turns.prefix_text.iloc[5], 4) == turns.prefix_text.iloc[5]
print(turns.prefix_text.iloc[5][:400])

## 4. External conversations (volume for the encoder)

Our study contributes 1,479 real in-domain turns (1,080 primary). A 67M encoder needs more text than that to
learn the rubric, so we weak-label public multi-turn chat logs with the same judge. Order of preference:
`allenai/WildChat-1M` (gated: accept the terms on Hugging Face and set `HF_TOKEN`), `lmsys/lmsys-chat-1m` (gated),
`OpenAssistant/oasst1` (open, human-written prompts, used as fallback). Prefixes are cut at a random user turn
\(k\le 6\) so turn positions beyond our 4-turn study appear in training.

In [ ]:
# ---- 4. External corpora -----------------------------------------------------
from datasets import load_dataset

def _prefixes_from_conversation(msgs, n_max=2, rng=random):
    # msgs: list of {"role","content"} alternating; yield up to n_max random user-turn prefixes
    user_idx = [i for i, m in enumerate(msgs) if m["role"] == "user" and m["content"].strip()]
    user_idx = [i for i in user_idx if i <= 11]           # k <= 6
    if not user_idx:
        return []
    picks = rng.sample(user_idx, k=min(n_max, len(user_idx)))
    out = []
    for i in picks:
        k = sum(1 for j in user_idx if j <= i)
        out.append({"turn": k, "user_text": msgs[i]["content"],
                    "prefix_text": serialise_prefix(msgs[: i + 1], k)})
    return out

def load_wildchat(n):
    ds = load_dataset("allenai/WildChat-1M", split="train", streaming=True)
    rows = []
    for ex in ds:
        if ex.get("language") != "English" or ex.get("toxic"):
            continue
        msgs = [{"role": m["role"], "content": m["content"]} for m in ex["conversation"] if m["role"] in ("user", "assistant")]
        if sum(m["role"] == "user" for m in msgs) < 1:
            continue
        rows.extend(_prefixes_from_conversation(msgs))
        if len(rows) >= n:
            break
    return rows[:n], "wildchat"

def load_lmsys(n):
    ds = load_dataset("lmsys/lmsys-chat-1m", split="train", streaming=True)
    rows = []
    for ex in ds:
        if ex.get("language") != "English" or any(m.get("flagged") for m in (ex.get("openai_moderation") or [])):
            continue
        msgs = [{"role": m["role"], "content": m["content"]} for m in ex["conversation"] if m["role"] in ("user", "assistant")]
        rows.extend(_prefixes_from_conversation(msgs))
        if len(rows) >= n:
            break
    return rows[:n], "lmsys"

def load_oasst1(n):
    ds = load_dataset("OpenAssistant/oasst1", split="train")
    df = ds.to_pandas()
    df = df[df.lang == "en"]
    by_id = df.set_index("message_id")
    children = df.groupby("parent_id").message_id.apply(list).to_dict()
    roots = df[df.parent_id.isna()].message_id.tolist()
    random.shuffle(roots)
    rows = []
    role = {"prompter": "user", "assistant": "assistant"}
    for r in roots:
        msgs, cur = [], r
        while cur is not None:
            m = by_id.loc[cur]
            msgs.append({"role": role[m.role], "content": m.text})
            kids = children.get(cur)
            cur = random.choice(kids) if kids else None
        rows.extend(_prefixes_from_conversation(msgs))
        if len(rows) >= n:
            break
    return rows[:n], "oasst1"

def load_external(n, preferred):
    order = {"wildchat": [load_wildchat, load_lmsys, load_oasst1],
             "lmsys": [load_lmsys, load_wildchat, load_oasst1],
             "oasst1": [load_oasst1]}[preferred]
    for fn in order:
        try:
            rows, name = fn(n)
            if rows:
                return rows, name
        except Exception as e:                       # gated dataset without token, network, schema drift
            print(f"{fn.__name__} unavailable: {type(e).__name__}: {str(e)[:120]}")
    raise RuntimeError("no external corpus could be loaded")

ext_path = WORK / "external_prefixes.csv"
if ext_path.exists():
    external = pd.read_csv(ext_path)
else:
    rows, name = load_external(CFG["n_external"], CFG["external_source"])
    external = pd.DataFrame(rows)
    external["source_set"] = "external"; external["corpus"] = name
    external["row_id"] = [f"ext_{i}" for i in range(len(external))]
    external.to_csv(ext_path, index=False)
print(external.corpus.iloc[0], external.shape, external.turn.value_counts().sort_index().to_dict())

## 5. The judge: rubric and weak labels

Four items (need, readiness, safety, fit), each on an anchored 0-9 scale. The judge is a local instruction model and the
score is the **expected value over its next-token distribution on the ten digits**, so the label is continuous and
carries uncertainty. Version 1 (0/1/2 integers generated as JSON) saturated on the study turns (mean 0.98) and was
revised once; v2 is frozen. The rubric *is* the definition of receptivity, so the next section checks it against things
the judge never saw: ThradBERT's `purchasable_products` posterior and the human anchor. Item 3 (`safe`) also exists as a
hard veto rule; the learned head is only a soft duplicate for transparency.

In [ ]:
# ---- 5. Judge ----------------------------------------------------------------
# v1 (0/1/2 integers in JSON, generated) saturated on the study turns: mean r = 0.98,
# three quarters of rows at the ceiling on every item. Revised once, then frozen:
# v2 scores each item on a 0-9 anchored scale and reads the *expected value* over the
# digit-token probabilities (one forward pass per item, no generation), so the label is
# continuous and carries the judge's uncertainty instead of its argmax.
RUBRIC_VERSION = "v2_logprob_0to9"

CONTEXT = ("You evaluate whether NOW is a good moment for a chatbot to include one short, honest, relevant "
           "product or service suggestion in its next reply. You see the conversation so far, ending with the "
           "user's latest message. Rate exactly one aspect on a 0-9 scale using the anchors given. Reply with a single digit.")

ITEM_PROMPTS = {
    "need": ("NEED. Is there a need, task, purchase or decision that a product or service could help with? "
             "0 = none. 3 = the topic has products around it but the user did not ask about any. "
             "6 = the user works on a task where a product would help but has not asked what to get. "
             "9 = the user explicitly asks what to buy, which product, or for options."),
    "ready": ("READINESS. Is this the right turn, or should the assistant wait? "
               "0 = mid-task (debugging, editing their text, a narrow follow-up) or a specific non-commercial question. "
               "5 = the user is still exploring; a suggestion would not derail but is not asked for. "
               "9 = the user is at a decision point asking for options or recommendations right now."),
    "safe": ("SAFETY. Is the topic free of sensitive content? "
             "0 = medical or mental health, grief, relationships, legal trouble, sexual content, minors, illegal acts, self-harm. "
             "5 = mildly personal or emotional but not sensitive. "
             "9 = fully safe practical or commercial topic."),
    "no_degrade": ("FIT. Could one brief product mention be added to the assistant's next reply without hurting its quality? "
                   "0 = it would damage the answer (off-topic, dishonest, or the user asked for neutral information). "
                   "5 = it would fit only as an aside. "
                   "9 = a product is the natural answer to what the user asked."),
}
ITEMS = list(ITEM_PROMPTS)
DIGITS = [str(d) for d in range(10)]

class Judge:
    def __init__(self, model_name, max_prompt_tokens=900):
        self.tok = AutoTokenizer.from_pretrained(model_name, padding_side="left")
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        dtype = AMP_DTYPE if DEVICE == "cuda" else torch.float32
        try:
            self.model = AutoModelForCausalLM.from_pretrained(model_name, dtype=dtype)
        except TypeError:
            self.model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype)
        self.model = self.model.to(DEVICE).eval()
        self.max_prompt_tokens = max_prompt_tokens
        ids = [self.tok.encode(d, add_special_tokens=False) for d in DIGITS]
        assert all(len(i) == 1 for i in ids), "digits must be single tokens for this tokenizer"
        self.digit_ids = torch.tensor([i[0] for i in ids], device=DEVICE)
        self.digit_values = torch.arange(10, device=DEVICE, dtype=torch.float32) / 9.0

    def _prompt(self, prefix_text, item):
        ids = self.tok(prefix_text, add_special_tokens=False).input_ids
        if len(ids) > self.max_prompt_tokens:
            prefix_text = self.tok.decode(ids[-self.max_prompt_tokens:])
        msgs = [{"role": "system", "content": CONTEXT + "\n\n" + ITEM_PROMPTS[item]},
                {"role": "user", "content": "Conversation:\n" + prefix_text + "\n\nRating (single digit 0-9):"}]
        try:
            return self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        except TypeError:
            return self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    @torch.no_grad()
    def score_item(self, texts, item, batch_size=24):
        out = []
        for i in range(0, len(texts), batch_size):
            batch = [self._prompt(t, item) for t in texts[i:i + batch_size]]
            enc = self.tok(batch, return_tensors="pt", padding=True).to(DEVICE)
            logits = self.model(**enc).logits[:, -1, :]                  # next-token distribution
            probs = torch.softmax(logits[:, self.digit_ids].float(), dim=-1)
            out.append((probs @ self.digit_values).cpu().numpy())        # expected value in [0,1]
        return np.concatenate(out)

    def score(self, texts, items=ITEMS, batch_size=24, desc="judge"):
        res = {}
        for item in items:
            t0 = time.time(); res[item] = self.score_item(texts, item, batch_size)
            print(f"{desc}: {item} done in {(time.time()-t0)/60:.1f} min")
        return pd.DataFrame(res)

def judge_frame(judge, df, cache_path, text_col="prefix_text"):
    if cache_path.exists():
        cached = pd.read_csv(cache_path)
        ver = cached["rubric_version"].iloc[0] if "rubric_version" in cached.columns else ""
        if len(cached) == len(df) and ver == RUBRIC_VERSION:
            return cached
        print("cache miss or stale rubric:", cache_path.name, "got", ver, "want", RUBRIC_VERSION)
    lab = judge.score(df[text_col].tolist(), batch_size=CFG["judge_batch"] if DEVICE == "cuda" else 2, desc=cache_path.stem)
    lab["parse_ok"] = True
    lab["r_mean"] = lab[ITEMS].mean(axis=1)
    lab["rubric_version"] = RUBRIC_VERSION
    lab.insert(0, "row_id", df["row_id"].values)
    lab.to_csv(cache_path, index=False)
    return lab

turns["row_id"] = [f"real_{i}" for i in range(len(turns))]
t0 = time.time()
judge = Judge(CFG["judge_model"])
lab_real = judge_frame(judge, turns, WORK / f"labels_real_{RUBRIC_VERSION}.csv")
lab_ext = judge_frame(judge, external, WORK / f"labels_external_{RUBRIC_VERSION}.csv")
print(f"judge done in {(time.time()-t0)/60:.1f} min")
print(lab_real[ITEMS + ["r_mean"]].describe().round(3))
print("share of real rows with r_mean > 0.9:", round(float((lab_real.r_mean > 0.9).mean()), 3))


In [ ]:
# ---- 5b. Judge self-consistency: sensitivity to the prompt wording -----------------------
# Re-score a sample with the anchors removed (bare 0-9 question). If the anchored and bare scores
# disagree, the label is carried by the anchors, which is what we want to know before trusting it.
from scipy.stats import spearmanr as _spearmanr

def spearman(a, b):
    # Spearman rho that returns nan (no warning) when either input is constant
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    ok = ~(np.isnan(a) | np.isnan(b))
    if ok.sum() < 3 or np.nanstd(a[ok]) == 0 or np.nanstd(b[ok]) == 0:
        return float("nan")
    return float(_spearmanr(a[ok], b[ok]).statistic)

audit_n = min(CFG["judge_audit_n"], len(turns))
audit_idx = np.random.RandomState(SEED).choice(len(turns), audit_n, replace=False)
audit_texts = turns.prefix_text.iloc[audit_idx].tolist()
_saved = dict(ITEM_PROMPTS)
for k in ITEM_PROMPTS:                                   # strip anchors, keep the question
    ITEM_PROMPTS[k] = _saved[k].split(" 0 = ")[0] + " Rate 0 (no) to 9 (yes)."
bare = judge.score(audit_texts, desc="audit-bare")
ITEM_PROMPTS.update(_saved)
orig = lab_real.iloc[audit_idx].reset_index(drop=True)
agree = {k: round(spearman(orig[k], bare[k]), 3) for k in ITEMS}
agree["r_mean"] = round(spearman(orig.r_mean, bare[ITEMS].mean(axis=1)), 3)
print("anchored vs bare rubric, Spearman:", agree)
json.dump(agree, open(WORK / "judge_self_agreement.json", "w"), indent=2)
del judge; torch.cuda.empty_cache() if DEVICE == "cuda" else None

## 6. Does the rubric mean anything? Checks against signals the judge never saw

1. \(\bar r\) vs ThradBERT \(p(\texttt{purchasable\_products})\) on the same bare utterance.
2. \(\bar r\) at the advertised turn vs the human anchor: \(U_i^{\text{resid}}\) (Spearman) and `good_moment_human` (AUROC).
3. \(\bar r\) at the advertised turn vs cued `recall_memory`.
4. Descriptives by turn and by task genre (Informational / Transactional / Social).

If none of these move, the rubric is revised **once** and frozen; the notebook records the numbers either way.

In [ ]:
# ---- 6. Judge validation --------------------------------------------------------
from sklearn.metrics import roc_auc_score
real = turns.merge(lab_real, on="row_id")
prim = real[real.source_set == "primary"].copy()

checks = {}
if "p_purchasable" in prim and prim.p_purchasable.notna().any():
    checks["spearman_rmean_vs_p_purchasable"] = round(spearman(prim.r_mean, prim.p_purchasable), 3)
    checks["spearman_need_vs_p_purchasable"] = round(spearman(prim.need, prim.p_purchasable), 3)

ad_rows = prim[prim.is_ad_turn == 1][["conversation_id", "turn", "r_mean", *ITEMS]]
anc = anchor.merge(ad_rows, on="conversation_id", how="left")
ok = anc.r_mean.notna()
checks["anchor_rows_with_judge"] = int(ok.sum())
checks["spearman_rmean_vs_ux_retention_resid"] = round(spearman(anc.r_mean[ok], anc.ux_retention_resid[ok]), 3)
checks["auroc_rmean_vs_good_moment_human"] = round(float(roc_auc_score(anc.good_moment_human[ok], anc.r_mean[ok])), 3)
checks["spearman_rmean_vs_recall_memory"] = round(spearman(anc.r_mean[ok], anc.recall_memory[ok]), 3)
for k in ITEMS:
    checks[f"spearman_{k}_vs_ux_retention_resid"] = round(spearman(anc[k][ok], anc.ux_retention_resid[ok]), 3)

print(json.dumps(checks, indent=2))
print("\nr_mean by turn (primary):"); print(prim.groupby("turn").r_mean.agg(["mean", "std"]).round(3))
print("\nr_mean by task genre:"); print(prim.groupby("task_genre").r_mean.mean().round(3))
print("\nUX retention by timing (anchor):"); print(anc.groupby("timing").ux_retention_resid.agg(["mean", "count"]).round(3))
json.dump(checks, open(WORK / "judge_validation.json", "w"), indent=2)

## 7. Encoder: fine-tune a 67M DistilBERT with a 4-output regression head

Base model: the ThradBERT backbone (`Thrad/thrad-bert-conversation-classifier`, DistilBERT, 512 tokens,
<50 ms per prefix on CPU). It is already the project's \(f_{\mathrm{genre}}\), so its representation has seen
exactly this kind of text. The 13-way head is discarded; a fresh `Linear(768, 4)` head predicts the rubric.
Loss: soft-target binary cross-entropy per item. Real rows are split by **participant** (GroupKFold);
external rows always train. Reported metric: Spearman(\(\hat{\bar r}\), \(\bar r\)) on held-out real rows.

In [ ]:
# ---- 7. Encoder training ---------------------------------------------------------
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold

class PrefixDataset(Dataset):
    def __init__(self, texts, targets, tok, max_len):
        self.enc = tok(list(texts), truncation=True, max_length=max_len, padding=False)
        self.t = torch.tensor(np.asarray(targets, dtype=np.float32))
    def __len__(self): return len(self.t)
    def __getitem__(self, i):
        return {"input_ids": self.enc["input_ids"][i], "attention_mask": self.enc["attention_mask"][i], "labels": self.t[i]}

def collate(batch, pad_id):
    L = max(len(b["input_ids"]) for b in batch)
    ids = torch.full((len(batch), L), pad_id, dtype=torch.long); am = torch.zeros((len(batch), L), dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["input_ids"]); ids[i, :n] = torch.tensor(b["input_ids"]); am[i, :n] = 1
    return ids, am, torch.stack([b["labels"] for b in batch])

class MomentEncoder(nn.Module):
    def __init__(self, name, n_out=4):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(name)
        h = self.backbone.config.hidden_size
        self.head = nn.Sequential(nn.Dropout(0.1), nn.Linear(h, n_out))
    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        return self.head(out[:, 0])          # [CLS] logits, 4 items

def tokenizer_for(name):
    try:
        return AutoTokenizer.from_pretrained(name)
    except Exception:
        return AutoTokenizer.from_pretrained("bert-base-uncased")   # ThradBERT ships without tokenizer files

def train_encoder(name, train_texts, train_y, val_texts, val_y, epochs, lr, bs, max_len, max_steps=None, log=True):
    tok = tokenizer_for(name)
    model = MomentEncoder(name).to(DEVICE)
    tr = DataLoader(PrefixDataset(train_texts, train_y, tok, max_len), batch_size=bs, shuffle=True,
                    collate_fn=lambda b: collate(b, tok.pad_token_id))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total = epochs * len(tr) if max_steps is None else min(max_steps, epochs * len(tr))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / max(1, int(0.1 * total))) * max(0.0, (total - s) / total))
    scaler = torch.amp.GradScaler(enabled=(DEVICE == "cuda" and AMP_DTYPE == torch.float16))
    loss_fn = nn.BCEWithLogitsLoss()
    step, history = 0, []
    for ep in range(epochs):
        model.train()
        for ids, am, y in tr:
            ids, am, y = ids.to(DEVICE), am.to(DEVICE), y.to(DEVICE)
            with torch.autocast(device_type="cuda" if DEVICE == "cuda" else "cpu", dtype=AMP_DTYPE, enabled=(DEVICE == "cuda")):
                logits = model(ids, am)
            loss = loss_fn(logits.float(), y)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step()
            step += 1
            if max_steps and step >= max_steps:
                break
        m = evaluate(model, tok, val_texts, val_y, max_len) if len(val_texts) else {}
        history.append({"epoch": ep, "loss": loss.item(), **m})
        if log: print(f"  epoch {ep}: loss {loss.item():.4f} {m}")
        if max_steps and step >= max_steps:
            break
    return model, tok, history

@torch.no_grad()
def predict(model, tok, texts, max_len, bs=64):
    model.eval(); outs = []
    for i in range(0, len(texts), bs):
        enc = tok(list(texts[i:i + bs]), truncation=True, max_length=max_len, padding=True, return_tensors="pt").to(DEVICE)
        with torch.autocast(device_type="cuda" if DEVICE == "cuda" else "cpu", dtype=AMP_DTYPE, enabled=(DEVICE == "cuda")):
            outs.append(torch.sigmoid(model(enc.input_ids, enc.attention_mask).float()).cpu().numpy())
    return np.concatenate(outs) if outs else np.zeros((0, 4))

def evaluate(model, tok, texts, y, max_len):
    p = predict(model, tok, texts, max_len)
    y = np.asarray(y)
    return {"spearman_rmean": round(spearman(p.mean(1), y.mean(1)), 3),
            "mae_rmean": round(float(np.abs(p.mean(1) - y.mean(1)).mean()), 3),
            **{f"spearman_{k}": round(spearman(p[:, j], y[:, j]), 3) for j, k in enumerate(ITEMS)}}

# assemble labelled frames
real_lab = real[real.parse_ok].copy()
ext_lab = external.merge(lab_ext, on="row_id"); ext_lab = ext_lab[ext_lab.parse_ok].copy()
print("labelled real", len(real_lab), "external", len(ext_lab))

In [ ]:
# ---- 7b. Cross-validated encoder by participant + ablations ------------------------
def run_cv(name, window, use_external, folds, epochs, tag):
    gkf = GroupKFold(n_splits=folds)
    oof = pd.Series(np.nan, index=real_lab.index)
    hist = []
    Xr = real_lab.prefix_text.map(lambda s: reserialise(s, window)) if window != 4 else real_lab.prefix_text
    Xe = ext_lab.prefix_text.map(lambda s: reserialise(s, window)) if window != 4 else ext_lab.prefix_text
    for f, (tr_i, va_i) in enumerate(gkf.split(real_lab, groups=real_lab.fold_participant)):
        tr_texts = list(Xr.iloc[tr_i]) + (list(Xe) if use_external else [])
        tr_y = np.vstack([real_lab[ITEMS].iloc[tr_i].to_numpy()] + ([ext_lab[ITEMS].to_numpy()] if use_external else []))
        model, tok, h = train_encoder(name, tr_texts, tr_y, list(Xr.iloc[va_i]), real_lab[ITEMS].iloc[va_i].to_numpy(),
                                      epochs, CFG["lr"], CFG["batch_size"], CFG["max_len"], CFG["max_train_steps"], log=False)
        p = predict(model, tok, list(Xr.iloc[va_i]), CFG["max_len"])
        oof.iloc[va_i] = p.mean(1)
        hist.append(h[-1]); del model; torch.cuda.empty_cache() if DEVICE == "cuda" else None
        print(f"[{tag}] fold {f}: {h[-1]}")
    summary = {"tag": tag, "encoder": name, "window": window, "external": use_external,
               "spearman_oof": round(spearman(oof, real_lab.r_mean), 3),
               "mae_oof": round(float(np.abs(oof - real_lab.r_mean).mean()), 3)}
    return oof, summary

results, oof_store = [], {}
for window in CFG["ablation_windows"]:
    for use_ext in CFG["ablation_external"]:
        tag = f"{Path(CFG['encoder']).name}_w{window}_ext{int(use_ext)}"
        oof, s = run_cv(CFG["encoder"], window, use_ext, CFG["cv_folds"], CFG["epochs"], tag)
        results.append(s); oof_store[tag] = oof
for b in CFG["baselines"]:
    tag = f"{Path(b).name}_w4_ext1"
    oof, s = run_cv(b, 4, True, CFG["cv_folds"], CFG["epochs"], tag)
    results.append(s); oof_store[tag] = oof

# tabular baseline: turn index + runtime genre one-hot -> r_mean (what the text adds over metadata)
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_predict
Xtab = pd.get_dummies(real_lab[["intent_runtime"]].fillna("none")).assign(turn=real_lab.turn.clip(upper=8) / 8, msg=np.log1p(real_lab.msg_len.fillna(0)))
oof_tab = cross_val_predict(Ridge(alpha=1.0), Xtab.to_numpy(dtype=float), real_lab.r_mean, groups=real_lab.fold_participant, cv=GroupKFold(CFG["cv_folds"]))
results.append({"tag": "tabular_ridge", "encoder": "none", "window": 0, "external": False,
                "spearman_oof": round(spearman(oof_tab, real_lab.r_mean), 3),
                "mae_oof": round(float(np.abs(oof_tab - real_lab.r_mean).mean()), 3)})
results_df = pd.DataFrame(results); results_df.to_csv(WORK / "ablations.csv", index=False)
print(results_df.to_string(index=False))
PRIMARY_TAG = f"{Path(CFG['encoder']).name}_w4_ext1"

## 8. Calibration to the human anchor and the non-inferiority check

The encoder predicts the judge. The human anchor asks whether that has anything to do with people.
Out-of-fold \(\hat{\bar r}\) at each advertised turn (never seen by the fold that predicts it) plus the two context
scalars feed a logistic calibrator fitted leave-one-participant-out. Then the within-person check: split each
participant's four advertised turns at their own median \(\hat{\bar r}\); is UX retention at high-score moments
*not worse* than at low-score moments? Margin: 0.5 Likert points.

In [ ]:
# ---- 8. Calibration + non-inferiority ------------------------------------------------
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneGroupOut
from scipy import stats

oof_primary = oof_store[PRIMARY_TAG]
ad_pred = real_lab.loc[real_lab.is_ad_turn == 1, ["conversation_id", "turn", "fit_score"]].assign(r_hat=oof_primary[real_lab.is_ad_turn == 1].values)
cal = anchor.merge(ad_pred, on="conversation_id", how="inner", suffixes=("", "_pred"))
cal["turn_feat"] = cal.ad_turn.clip(upper=8) / 8
cal["fit_feat"] = cal.fit_score.fillna(cal.fit_score.median()).clip(0, 1)
Xc = cal[["r_hat", "turn_feat", "fit_feat"]].to_numpy(dtype=float)
yc = cal.good_moment_human.to_numpy()

logo = LeaveOneGroupOut()
p_oof = np.zeros(len(cal))
for tr_i, te_i in logo.split(Xc, yc, groups=cal.fold_participant):
    clf = LogisticRegression(C=1.0, max_iter=500).fit(Xc[tr_i], yc[tr_i])
    p_oof[te_i] = clf.predict_proba(Xc[te_i])[:, 1]
calibrator = LogisticRegression(C=1.0, max_iter=500).fit(Xc, yc)
cal["moment_score"] = p_oof

calib = {
    "n": int(len(cal)),
    "auroc_calibrated_lopo": round(float(roc_auc_score(yc, p_oof)), 3),
    "auroc_r_hat_only": round(float(roc_auc_score(yc, cal.r_hat)), 3),
    "auroc_turn_only": round(float(roc_auc_score(yc, cal.turn_feat)), 3),
    "spearman_r_hat_vs_ux_resid": round(spearman(cal.r_hat, cal.ux_retention_resid), 3),
    "coef": dict(zip(["r_hat", "turn_feat", "fit_feat"], np.round(calibrator.coef_[0], 3).tolist())),
    "intercept": round(float(calibrator.intercept_[0]), 3),
}

# within-person high vs low r_hat -> UX retention (paired), non-inferiority margin 0.5
rows = []
for pid, g in cal.groupby("fold_participant"):
    if len(g) < 2: continue
    med = g.r_hat.median(); hi, lo = g[g.r_hat > med], g[g.r_hat <= med]
    if len(hi) and len(lo):
        rows.append(hi.ux_retention_resid.mean() - lo.ux_retention_resid.mean())
d = np.asarray(rows); MARGIN = -0.5
ci = stats.t.interval(0.95, len(d) - 1, loc=d.mean(), scale=stats.sem(d)) if len(d) > 2 else (np.nan, np.nan)
calib["noninferiority"] = {"participants": int(len(d)), "mean_diff_high_minus_low": round(float(d.mean()), 3),
                           "ci95": [round(float(ci[0]), 3), round(float(ci[1]), 3)], "margin": MARGIN,
                           "non_inferior": bool(ci[0] > MARGIN) if len(d) > 2 else None,
                           "wilcoxon_p_raw": round(float(stats.wilcoxon(d).pvalue), 4) if len(d) > 5 and np.any(d != 0) else None}
# structural reference: early vs late in the same anchor
e = cal[cal.timing == "early"].groupby("fold_participant").ux_retention_resid.mean()
l = cal[cal.timing == "late"].groupby("fold_participant").ux_retention_resid.mean()
common = e.index.intersection(l.index)
calib["early_minus_late_ux_resid"] = round(float((e[common] - l[common]).mean()), 3)
# negative controls: permutation null for the AUROC (200 global shuffles, and 200 within-participant shuffles)
rng = np.random.RandomState(SEED)
null_global = [roc_auc_score(rng.permutation(yc), p_oof) for _ in range(200)]
null_within = []
for _ in range(200):
    ys = cal.groupby("fold_participant").good_moment_human.transform(lambda s: rng.permutation(s.values))
    if ys.nunique() > 1:
        null_within.append(roc_auc_score(ys, p_oof))
calib["auroc_null_global"] = {"mean": round(float(np.mean(null_global)), 3), "p2.5": round(float(np.percentile(null_global, 2.5)), 3), "p97.5": round(float(np.percentile(null_global, 97.5)), 3)}
calib["auroc_null_within_person"] = {"mean": round(float(np.mean(null_within)), 3), "p2.5": round(float(np.percentile(null_within, 2.5)), 3), "p97.5": round(float(np.percentile(null_within, 97.5)), 3)}
print(json.dumps(calib, indent=2)); json.dump(calib, open(WORK / "calibration.json", "w"), indent=2)

## 8b. Optional UX multi-output head (diagnostic, not the gate)

Frozen encoder, 216 ad-turn prefixes only. Predicts
\(\mathbf y = (\Delta\mathrm{cred},\Delta\mathrm{trust},\Delta\mathrm{manip},\Delta\mathrm{help},\mathrm{notice}_{\mathrm{sponsored}},\mathrm{recall}_{\mathrm{memory}})\).
Leave-one-participant-out. Honest prior from smoke: near-null. This head never writes `moment_score`.


In [ ]:
# ---- 8b. UX diagnostic head ------------------------------------------------
UX_REPORT = {"trained": False}
if CFG.get("train_ux_head") and PRIMARY_TAG in oof_store:
    from sklearn.linear_model import Ridge
    from sklearn.model_selection import LeaveOneGroupOut

    ad = real_lab[real_lab.is_ad_turn == 1][["conversation_id", "prefix_text", "fold_participant"]].copy()
    ux = anchor.merge(ad, on="conversation_id", how="inner")
    for c in UX_COLS:
        ux[c] = pd.to_numeric(ux[c], errors="coerce")
    ux = ux.dropna(subset=UX_COLS)
    print("UX-head rows", len(ux), "participants", ux.fold_participant.nunique())

    # frozen [CLS] from the just-trained final-fold-style encoder: embed with a freshly loaded frozen backbone
    tok_ux = tokenizer_for(CFG["encoder"])
    frozen = AutoModel.from_pretrained(CFG["encoder"]).to(DEVICE).eval()

    @torch.no_grad()
    def embed(texts, max_len=CFG["max_len"], bs=32):
        outs = []
        for i in range(0, len(texts), bs):
            enc = tok_ux(list(texts[i:i + bs]), truncation=True, max_length=max_len, padding=True, return_tensors="pt").to(DEVICE)
            with torch.autocast(device_type="cuda" if DEVICE == "cuda" else "cpu", dtype=AMP_DTYPE, enabled=(DEVICE == "cuda")):
                h = frozen(enc.input_ids, enc.attention_mask).last_hidden_state[:, 0].float()
            outs.append(h.cpu().numpy())
        return np.concatenate(outs)

    H = embed(ux.prefix_text.tolist())
    Y = ux[UX_COLS].to_numpy(dtype=float)
    # standardise Y for the ridge; report back in original units
    y_mean, y_std = Y.mean(0), Y.std(0) + 1e-6
    Yz = (Y - y_mean) / y_std
    logo = LeaveOneGroupOut()
    pred = np.zeros_like(Y)
    for tr, te in logo.split(H, Yz, groups=ux.fold_participant):
        clf = Ridge(alpha=10.0).fit(H[tr], Yz[tr])
        pred[te] = clf.predict(H[te]) * y_std + y_mean
    UX_REPORT = {
        "trained": True,
        "n": int(len(ux)),
        "spearman": {c: round(spearman(pred[:, j], Y[:, j]), 3) for j, c in enumerate(UX_COLS)},
        "mae": {c: round(float(np.abs(pred[:, j] - Y[:, j]).mean()), 3) for j, c in enumerate(UX_COLS)},
        "note": "frozen ThradBERT [CLS] + ridge, LOPO by participant; not the serving gate",
    }
    # fit a single head on all 216 for export
    ux_ridge = Ridge(alpha=10.0).fit(H, Yz)
    UX_EXPORT = {
        "mean": y_mean.tolist(), "std": y_std.tolist(),
        "coef": ux_ridge.coef_.tolist(), "intercept": ux_ridge.intercept_.tolist(),
        "cols": UX_COLS,
    }
    json.dump(UX_REPORT, open(WORK / "ux_head.json", "w"), indent=2)
    json.dump(UX_EXPORT, open(WORK / "ux_head_weights.json", "w"), indent=2)
    print(json.dumps(UX_REPORT, indent=2))
    del frozen
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
else:
    print("UX head skipped (smoke or disabled)")


## 9. Final model, export, and the serving contract

Train on every labelled row (real + external), save weights + tokenizer + calibrator + serialisation config,
and wrap them in `MomentScorer`, whose `score()` accepts exactly the production schema:
`messages`, `turn_index`, `turns_since_last_ad`, optional `draft_reply`, optional `candidate.fit_score`.
The governance veto uses the runtime ThradBERT label when the caller passes one (`genre_label`), so no second
model is required at serving time.

In [ ]:
# ---- 9. Final fit + export -------------------------------------------------------------
final_texts = list(real_lab.prefix_text) + list(ext_lab.prefix_text)
final_y = np.vstack([real_lab[ITEMS].to_numpy(), ext_lab[ITEMS].to_numpy()])
final_model, final_tok, _ = train_encoder(CFG["encoder"], final_texts, final_y, [], [], CFG["epochs"], CFG["lr"],
                                          CFG["batch_size"], CFG["max_len"], CFG["max_train_steps"], log=False)
EXPORT = WORK / "moment_scorer_model"; EXPORT.mkdir(exist_ok=True)
final_model.backbone.save_pretrained(EXPORT / "backbone"); final_tok.save_pretrained(EXPORT / "backbone")
torch.save(final_model.head.state_dict(), EXPORT / "head.pt")
default_tau = float(np.quantile(p_oof, 1 - yc.mean()))      # serve about as often as humans were fine with it
json.dump({
    "items": ITEMS, "max_prefix_turns": MAX_PREFIX_TURNS, "max_len": CFG["max_len"], "role_tags": ROLE_TAGS,
    "calibrator": {"features": ["r_hat", "turn_feat", "fit_feat"], "coef": calibrator.coef_[0].tolist(),
                   "intercept": float(calibrator.intercept_[0]), "fit_fill": float(cal.fit_score.median())},
    "tau_default": round(default_tau, 3), "veto_genres": sorted(VETO_GENRES), "anchor_sha256_16": anchor_hash,
    "encoder": CFG["encoder"], "run_mode": RUN_MODE, "env": ENV, "ablations": results, "calibration": calib,
    "ux_head": UX_REPORT, "rubric_version": "v2_logprob_0to9",
}, open(EXPORT / "moment_scorer_config.json", "w"), indent=2)
print("exported to", EXPORT, "tau_default", round(default_tau, 3))


In [ ]:
# ---- 9b. MomentScorer: the serving object -------------------------------------------------
class MomentScorer:
    def __init__(self, export_dir, device="cpu"):
        export_dir = Path(export_dir)
        self.cfg = json.load(open(export_dir / "moment_scorer_config.json"))
        self.tok = AutoTokenizer.from_pretrained(export_dir / "backbone")
        self.model = MomentEncoder(str(export_dir / "backbone"), n_out=len(self.cfg["items"]))
        self.model.head.load_state_dict(torch.load(export_dir / "head.pt", map_location="cpu"))
        self.model.to(device).eval(); self.device = device
        c = self.cfg["calibrator"]; self.w = np.asarray(c["coef"]); self.b = c["intercept"]; self.fit_fill = c["fit_fill"]
        ux_path = export_dir.parent / "ux_head_weights.json"
        self.ux = json.load(open(ux_path)) if ux_path.exists() else None

    @torch.no_grad()
    def score(self, messages, turn_index, turns_since_last_ad=None, draft_reply=None, candidate=None,
              genre_label=None, frequency_cap_turns=3, tau=None):
        veto = None
        if genre_label in set(self.cfg["veto_genres"]):
            veto = "sensitive_genre"
        elif turns_since_last_ad is not None and turns_since_last_ad < frequency_cap_turns:
            veto = "frequency_cap"
        msgs = list(messages) + ([{"role": "assistant", "content": draft_reply}] if draft_reply else [])
        text = serialise_prefix(msgs, turn_index, self.cfg["max_prefix_turns"])
        enc = self.tok(text, truncation=True, max_length=self.cfg["max_len"], return_tensors="pt").to(self.device)
        items = torch.sigmoid(self.model(enc.input_ids, enc.attention_mask).float()).cpu().numpy()[0]
        r_hat = float(items.mean())
        fit = float(candidate["fit_score"]) if candidate and candidate.get("fit_score") is not None else self.fit_fill
        z = np.array([r_hat, min(turn_index, 8) / 8, min(max(fit, 0.0), 1.0)])
        m = float(1 / (1 + np.exp(-(z @ self.w + self.b))))
        tau = self.cfg["tau_default"] if tau is None else tau
        out = {"moment_score": 0.0 if veto else round(m, 4), "receptivity": round(r_hat, 4),
                "items": {k: round(float(v), 3) for k, v in zip(self.cfg["items"], items)},
                "veto": veto, "insert": (veto is None) and m >= tau, "abstain": abs(m - tau) < 0.05}
        if self.ux is not None:
            # serving-time UX head needs the [CLS] of the *trained* backbone; skip if only ridge-on-frozen is stored
            out["ux_hat_note"] = "diagnostic weights in ux_head_weights.json; not used for insert"
        return out

scorer = MomentScorer(EXPORT, device="cpu")
demo = [
    ("commercial", [{"role": "user", "content": "I need a laptop for video editing under 1200 euros, what should I look for?"},
                    {"role": "assistant", "content": "Prioritise a recent CPU, 32 GB RAM and a colour-accurate display."},
                    {"role": "user", "content": "Ok, any specific models you would recommend?"}], None),
    ("emotional", [{"role": "user", "content": "My dog died yesterday and I cannot stop crying. How do I cope?"}], "relationships_and_personal_reflection"),
    ("coding", [{"role": "user", "content": "My pandas merge returns duplicated rows, here is the code: df.merge(other, on='id')"}], "programming_and_data_analysis"),
]
for name, msgs, genre in demo:
    k = sum(m["role"] == "user" for m in msgs)
    t0 = time.perf_counter(); out = scorer.score(msgs, k, genre_label=genre); dt = (time.perf_counter() - t0) * 1000
    print(f"{name:11s} {dt:6.1f} ms  {json.dumps(out)}")


## 10. What to read off this run

* `ablations.csv`: does text beat the tabular baseline; does external data help; which window.
* `judge_validation.json` and `judge_self_agreement.json`: is the rubric stable and does it track \(f_{\mathrm{genre}}\) and humans at all.
* `calibration.json`: AUROC against `good_moment_human` (leave-one-participant-out), the shuffled-label control, and the within-person non-inferiority interval. If the interval clears the margin but AUROC is near 0.5, the honest claim is "receptivity scorer, not worse for trust", not "predicts UX".
* `ux_head.json`: LOPO Spearman / MAE of the frozen-encoder UX vector. Near-zero is a result.
* `moment_scorer_model/`: drop-in for `src/project/core/policy/` as a gate before retrieval.

Support box for any release: four-turn scripted tasks, five scenarios, one conversational LLM, English, product exposure only, composites provisional until the Goal 1 freeze (`anchor_sha256_16` in the config pins the anchor version).

After the run, append the numbers to `.agents/context/data-analysis/policy/` as a new dated note. Do not put them in the thesis.
